In [0]:
USE CATALOG dbw_kroger_insights;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS total_views,
        SUM(carts)                                                             AS total_carts,
        SUM(purchases)                                                         AS total_purchases,
        ROUND(SUM(revenue), 2)                                                 AS revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        total_views,
        total_carts,
        total_purchases,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC)                              AS revenue_rank,
        COUNT(*)     OVER ()                                                   AS total_skus,
        SUM(revenue) OVER ()                                                   AS total_revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS cumulative_revenue
    FROM product_totals
)
SELECT 
    MAX(total_skus)                                                            AS total_skus,
    MAX(total_revenue)                                                         AS total_revenue,
    
    -- Question 1: Revenue share of the top 20% of SKUs (%)
    ROUND(
        SUM(CASE WHEN revenue_rank <= 0.2 * total_skus THEN revenue ELSE 0 END) * 100.0 / 
        MAX(total_revenue), 2
    )                                                                          AS top_20_sku_revenue_share_pct,
    
    -- Question 2: Percentage of SKUs needed to drive 80% of total revenue (%)
    ROUND(
        MIN(CASE WHEN cumulative_revenue >= 0.8 * total_revenue THEN revenue_rank END) * 100.0 / 
        MAX(total_skus), 2
    )                                                                          AS skus_needed_for_80_pct_revenue_pct
FROM ranked_products;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS total_views,
        SUM(carts)                                                             AS total_carts,
        SUM(purchases)                                                         AS total_purchases,
        ROUND(SUM(revenue), 2)                                                 AS revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        total_views,
        total_carts,
        total_purchases,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC)                              AS revenue_rank,
        COUNT(*)     OVER ()                                                   AS total_skus,
        SUM(revenue) OVER ()                                                   AS total_revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS cumulative_revenue
    FROM product_totals
)
SELECT 
    -- Catalog Health Metrics
    MAX(total_skus)                                                            AS total_skus,
    COUNT_IF(total_views > 0)                                                  AS active_skus,
    COUNT_IF(revenue > 0)                                                      AS selling_skus,
    COUNT_IF(total_views > 0 AND total_purchases = 0)                          AS zero_sale_skus,
    
    MAX(total_revenue)                                                         AS total_revenue,
    
    -- Pareto Concentration Metrics
    ROUND(
        SUM(CASE WHEN revenue_rank <= 0.2 * total_skus THEN revenue ELSE 0 END) * 100.0 / 
        MAX(total_revenue), 2
    )                                                                          AS top_20_sku_revenue_share_pct,
    
    ROUND(
        MIN(CASE WHEN cumulative_revenue >= 0.8 * total_revenue THEN revenue_rank END) * 100.0 / 
        MAX(total_skus), 2
    )                                                                          AS skus_needed_for_80_pct_revenue_pct
FROM ranked_products;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS total_views,
        SUM(carts)                                                             AS total_carts,
        SUM(purchases)                                                         AS total_purchases,
        ROUND(SUM(revenue), 2)                                                 AS revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    WHERE revenue > 0
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC)                              AS revenue_rank,
        COUNT(*)     OVER ()                                                   AS total_selling_skus,
        SUM(revenue) OVER ()                                                   AS total_revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS cumulative_revenue
    FROM product_totals
)
SELECT 
    MAX(total_selling_skus)                                                    AS total_selling_skus,
    MAX(total_revenue)                                                         AS total_revenue,
    
    -- What percentage of *selling* SKUs are needed to drive 80% of revenue?
    ROUND(
        MIN(CASE WHEN cumulative_revenue >= 0.8 * total_revenue THEN revenue_rank END) * 100.0 / 
        MAX(total_selling_skus), 2
    )                                                                          AS selling_skus_needed_for_80_pct_revenue_pct
FROM ranked_products;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS total_views,
        SUM(carts)                                                             AS total_carts,
        SUM(purchases)                                                         AS total_purchases,
        ROUND(SUM(revenue), 2)                                                 AS total_revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        total_views,
        total_carts,
        total_purchases,
        total_revenue,
        ROW_NUMBER() OVER (ORDER BY total_revenue DESC)                        AS revenue_rank
    FROM product_totals
)
SELECT 
    r.revenue_rank,
    r.product_id,
    p.category_code,
    p.brand,
    ROUND(p.median_price, 2)                                                   AS median_price,
    r.total_views,
    r.total_carts,
    r.total_purchases,
    ROUND(r.total_revenue, 2)                                                  AS total_revenue
FROM ranked_products r
LEFT JOIN dbw_kroger_insights.gold.ec_dim_product p
    ON r.product_id = p.product_id
WHERE r.revenue_rank <= 10
ORDER BY r.revenue_rank ASC;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(purchases)                                                         AS total_purchases
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
product_analysis AS (
    SELECT 
        CASE 
            WHEN pt.total_purchases > 0 THEN 'Selling SKUs' 
            ELSE 'Zero-Sellers' 
        END                                                                    AS sku_type,
        CASE 
            WHEN p.category_code IS NULL 
              OR p.category_code = 'unknown' 
              OR p.category_code = '' THEN 1 
            ELSE 0 
        END                                                                    AS is_missing_category
    FROM product_totals pt
    LEFT JOIN dbw_kroger_insights.gold.ec_dim_product p
        ON pt.product_id = p.product_id
)
SELECT 
    sku_type,
    COUNT(*)                                                                   AS total_skus,
    SUM(is_missing_category)                                                   AS missing_category_count,
    ROUND(SUM(is_missing_category) * 100.0 / COUNT(*), 2)                      AS missing_category_pct
FROM product_analysis
GROUP BY sku_type;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS total_views,
        SUM(carts)                                                             AS total_carts,
        SUM(purchases)                                                         AS total_purchases,
        ROUND(SUM(revenue), 2)                                                 AS total_revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        total_views,
        total_carts,
        total_purchases,
        total_revenue,
        ROW_NUMBER() OVER (ORDER BY total_revenue DESC)                        AS revenue_rank
    FROM product_totals
)
SELECT 
    r.revenue_rank,
    r.product_id,
    p.category_code,
    p.brand,
    ROUND(p.median_price, 2)                                                   AS median_price,
    r.total_views,
    r.total_carts,
    r.total_purchases,
    ROUND(r.total_revenue, 2)                                                  AS total_revenue
FROM ranked_products r
LEFT JOIN dbw_kroger_insights.gold.ec_dim_product p
    ON r.product_id = p.product_id
WHERE r.revenue_rank <= 10
ORDER BY r.revenue_rank ASC;

In [0]:
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS total_views,
        SUM(purchases)                                                         AS total_purchases
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
zero_seller_buckets AS (
    SELECT 
        product_id,
        total_views,
        CASE 
            WHEN total_views = 0 THEN '0 views'
            WHEN total_views BETWEEN 1 AND 9 THEN '1–9 views'
            WHEN total_views BETWEEN 10 AND 99 THEN '10–99 views'
            WHEN total_views BETWEEN 100 AND 999 THEN '100–999 views'
            ELSE '1000+ views'
        END AS view_bucket
    FROM product_totals
    WHERE total_purchases = 0
)
SELECT 
    view_bucket,
    COUNT(*)                                                                   AS zero_seller_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)                         AS pct_of_zero_sellers
FROM zero_seller_buckets
GROUP BY view_bucket
ORDER BY 
    CASE view_bucket 
        WHEN '0 views' THEN 1 
        WHEN '1–9 views' THEN 2 
        WHEN '10–99 views' THEN 3 
        WHEN '100–999 views' THEN 4 
        ELSE 5 
    END;

In [0]:
WITH product_totals AS (
    -- Step 1: Aggregate totals at the individual product level
    SELECT
        product_id,
        SUM(views) AS views,
        SUM(carts) AS carts,
        SUM(purchases) AS purchases,
        SUM(revenue) AS revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
category_aggregated AS (
    -- Step 2: Join with product dimension to get category_l1 and aggregate
    SELECT
        COALESCE(d.category_l1, 'Unknown') AS category_l1,
        COUNT(DISTINCT p.product_id) AS skus,
        COUNT(DISTINCT CASE WHEN p.purchases > 0 THEN p.product_id END) AS selling_skus,
        SUM(p.views) AS views,
        SUM(p.purchases) AS purchases,
        SUM(p.revenue) AS revenue
    FROM product_totals p
    LEFT JOIN dbw_kroger_insights.gold.ec_dim_product d 
        ON p.product_id = d.product_id
    GROUP BY COALESCE(d.category_l1, 'Unknown')
)
-- Step 3: Compute final percentages, ratios, and window functions
SELECT
    category_l1,
    skus,
    ROUND(100.0 * selling_skus / NULLIF(skus, 0), 2) AS selling_skus_pct,
    views,
    purchases,
    ROUND(revenue, 2) AS revenue,
    ROUND(100.0 * revenue / SUM(revenue) OVER (), 2) AS revenue_share_pct,
    ROUND(100.0 * purchases / NULLIF(views, 0), 4) AS view_to_purchase_pct,
    ROUND(revenue / NULLIF(views, 0), 2) AS revenue_per_view
FROM category_aggregated
ORDER BY revenue DESC;

In [0]:
WITH category_monthly_daily AS (
    SELECT
        COALESCE(p.category_l1, 'Unknown') AS category_l1,
        -- October Metrics (31 days)
        SUM(CASE WHEN d.year_month = '2019-10' THEN f.views ELSE 0 END) AS oct_views,
        SUM(CASE WHEN d.year_month = '2019-10' THEN f.purchases ELSE 0 END) AS oct_purchases,
        
        -- November Metrics (26 days, excluding Nov 14-17 anomaly window)
        SUM(CASE WHEN d.year_month = '2019-11' AND NOT (d.date BETWEEN DATE'2019-11-14' AND DATE'2019-11-17') THEN f.views ELSE 0 END) AS nov_views,
        SUM(CASE WHEN d.year_month = '2019-11' AND NOT (d.date BETWEEN DATE'2019-11-14' AND DATE'2019-11-17') THEN f.purchases ELSE 0 END) AS nov_purchases
    FROM dbw_kroger_insights.gold.ec_fact_product_daily f
    JOIN dbw_kroger_insights.gold.dim_date d
        ON f.event_date = d.date
    LEFT JOIN dbw_kroger_insights.gold.ec_dim_product p
        ON f.product_id = p.product_id
    WHERE d.year_month IN ('2019-10', '2019-11')
    GROUP BY COALESCE(p.category_l1, 'Unknown')
)
SELECT
    category_l1,
    -- Views per day (October / 31 days, November / 26 days)
    ROUND(oct_views * 1.0 / 31, 2) AS oct_views_per_day,
    ROUND(nov_views * 1.0 / 26, 2) AS nov_views_per_day,
    ROUND(
        (nov_views * 1.0 / 26 - oct_views * 1.0 / 31) * 100.0 / NULLIF(oct_views * 1.0 / 31, 0), 2
    ) AS views_change_pct,
    
    -- Conversion efficiency (View-to-purchase %)
    ROUND(100.0 * oct_purchases / NULLIF(oct_views, 0), 2) AS oct_view_to_purchase_pct,
    ROUND(100.0 * nov_purchases / NULLIF(nov_views, 0), 2) AS nov_view_to_purchase_pct,
    
    -- Change in percentage points
    ROUND(
        (100.0 * nov_purchases / NULLIF(nov_views, 0)) - (100.0 * oct_purchases / NULLIF(oct_views, 0)), 2
    ) AS conversion_change_pp
FROM category_monthly_daily
ORDER BY oct_views_per_day DESC;

In [0]:
CREATE OR REPLACE TABLE dbw_kroger_insights.gold.sku_segments 
COMMENT 'SKU-level performance segmentation table for merchandising, pricing, and catalog optimization workflows.'
AS
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS views,
        SUM(carts)                                                             AS carts,
        SUM(purchases)                                                         AS purchases,
        ROUND(SUM(revenue), 2)                                                 AS revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        views,
        carts,
        purchases,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC)                              AS revenue_rank,
        SUM(revenue) OVER (ORDER BY revenue DESC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS cumulative_revenue,
        SUM(revenue) OVER ()                                                   AS total_revenue
    FROM product_totals
),
with_cum_pct AS (
    SELECT
        product_id,
        views,
        carts,
        purchases,
        revenue,
        revenue_rank,
        -- Cumulative revenue percentage (0.0 to 1.0)
        CASE 
            WHEN total_revenue > 0 THEN cumulative_revenue / total_revenue 
            ELSE 0 
        END                                                                    AS cumulative_revenue_pct
    FROM ranked_products
),
segmented_products AS (
    SELECT
        w.product_id,
        COALESCE(p.category_l1, 'Unknown')                                     AS category_l1,
        p.brand,
        ROUND(p.median_price, 2)                                               AS median_price,
        w.views,
        w.purchases,
        ROUND(w.revenue, 2)                                                    AS revenue,
        w.revenue_rank,
        ROUND(w.cumulative_revenue_pct * 100.0, 2)                             AS cumulative_revenue_pct,
        
        -- Segment Rule Evaluation (ordered strictly by priority)
        CASE 
            WHEN w.purchases > 0 AND w.cumulative_revenue_pct <= 0.80 
                THEN 'Hero'
            WHEN w.purchases > 0 AND w.cumulative_revenue_pct > 0.80 
                THEN 'Core'
            WHEN w.purchases = 0 AND w.views >= 1000 
                THEN 'Fix'
            WHEN w.purchases = 0 AND w.views BETWEEN 100 AND 999 
                THEN 'Watch'
            ELSE 'Tail review'
        END                                                                    AS segment
    FROM with_cum_pct w
    LEFT JOIN dbw_kroger_insights.gold.ec_dim_product p
        ON w.product_id = p.product_id
)
SELECT
    product_id,
    category_l1,
    brand,
    median_price,
    views,
    purchases,
    revenue,
    revenue_rank,
    cumulative_revenue_pct,
    segment,
    -- Recommended Action mapping based on segment
    CASE segment
        WHEN 'Hero' THEN 'Protect availability and pricing'
        WHEN 'Core' THEN 'Maintain'
        WHEN 'Fix' THEN 'Review price, stock, and content'
        WHEN 'Watch' THEN 'Investigate by category'
        WHEN 'Tail review' THEN 'Delist or discoverability review'
        ELSE 'Review'
    END                                                                        AS recommended_action
FROM segmented_products;

In [0]:
SELECT
    segment,
    COUNT(*)                                                      AS skus,
    ROUND(SUM(revenue), 2)                                        AS revenue,
    ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 2)    AS revenue_share_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)            AS sku_share_pct
FROM gold.sku_segments
GROUP BY segment
ORDER BY revenue DESC;